# Exploratory Data Analysis (EDA) - Cardiovascular Disease

A simple, clean exploration of the **Cardiovascular Disease dataset** (`cardio_train.csv`).

### Objectives:
1. Inspect dataset structure, dimensions, and data types
2. Check for missing values and duplicates
3. Analyze target variable distribution (`cardio`)
4. Inspect summary statistics and physiological outliers (Blood Pressure, Height, Weight)
5. Examine categorical risk factors (Cholesterol, Glucose, Smoking, Alcohol, Activity)
6. Compute feature correlations with heart disease risk

## 1. Import Libraries & Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np

# Locate dataset path automatically
csv_path = "cardio_train.csv" if os.path.exists("cardio_train.csv") else os.path.join("backend", "cardio_train.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
sep = ";" if ";" in first_line else ","

df = pd.read_csv(csv_path, sep=sep)
print(f"Dataset loaded successfully from: {csv_path}")
df.head()

## 2. Dataset Dimensions & Information

In [ ]:
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\n--- Column Information & Data Types ---")
df.info()

## 3. Check for Missing Values & Duplicate Records

In [ ]:
# Missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)

# Duplicates
duplicate_count = df.duplicated().sum()
print(f"\nExact duplicate rows: {duplicate_count}")

## 4. Target Variable Distribution (`cardio`)
`0` = No Cardiovascular Disease, `1` = Cardiovascular Disease Present.

In [ ]:
target_counts = df["cardio"].value_counts()
target_pct = df["cardio"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage (%)": target_pct.round(2)
})
target_summary.index = ["0 (Healthy / Low Risk)", "1 (Cardiovascular Disease)"]
print("Target Distribution:")
target_summary

## 5. Summary Statistics of Numerical Features
Add `age_years` (age / 365.25) for easier clinical interpretation.

In [ ]:
df["age_years"] = (df["age"] / 365.25).round(1)

num_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo"]
df[num_cols].describe().round(2)

## 6. Outlier Inspection in Blood Pressure
Notice extreme values in raw data (e.g., negative BP, or systolic BP > 1,000 mmHg).

In [ ]:
print("Top 5 highest Systolic BP (ap_hi):")
print(df["ap_hi"].nlargest(5).values)

print("\nTop 5 lowest Systolic BP (ap_hi):")
print(df["ap_hi"].nsmallest(5).values)

# Physiological normal/realistic range: 50 <= ap_hi <= 250, 40 <= ap_lo <= 180
invalid_bp = df[
    (df["ap_hi"] < 50) | (df["ap_hi"] > 250) |
    (df["ap_lo"] < 40) | (df["ap_lo"] > 180) |
    (df["ap_hi"] < df["ap_lo"])
]
print(f"\nNumber of rows with extreme physiological BP outliers: {len(invalid_bp)} ({len(invalid_bp)/len(df)*100:.2f}%)")

## 7. Categorical Risk Factors vs. Cardiovascular Disease
Examine disease prevalence across categorical features:

In [ ]:
print("=== Cardiovascular Disease Rate by Cholesterol Level ===")
print(pd.crosstab(df["cholesterol"], df["cardio"], normalize="index").round(3) * 100)
print("\n1: Normal | 2: Above Normal | 3: Well Above Normal\n")

print("=== Cardiovascular Disease Rate by Glucose Level ===")
print(pd.crosstab(df["gluc"], df["cardio"], normalize="index").round(3) * 100)
print("\n1: Normal | 2: Above Normal | 3: Well Above Normal\n")

print("=== Cardiovascular Disease Rate by Physical Activity ===")
print(pd.crosstab(df["active"], df["cardio"], normalize="index").round(3) * 100)
print("\n0: Inactive | 1: Active\n")

print("=== Cardiovascular Disease Rate by Smoking ===")
print(pd.crosstab(df["smoke"], df["cardio"], normalize="index").round(3) * 100)
print("\n0: Non-Smoker | 1: Smoker")

## 8. Correlation with Target (`cardio`)

In [ ]:
corr = df.drop(columns=["id"]).corr()["cardio"].sort_values(ascending=False)
print("Correlation of each feature with Heart Disease (cardio):")
print(corr.round(4))

## 9. Key EDA Findings & Summary

1. **Balanced Dataset**: Target variable `cardio` is evenly distributed (~50% negative, ~50% positive). No severe class imbalance.
2. **Zero Missing Values**: The raw dataset contains complete data across all 70,000 rows.
3. **Strongest Predictors**:
   - **Systolic BP (`ap_hi`)** and **Diastolic BP (`ap_lo`)**
   - **Age** (risk increases substantially above age 50)
   - **Cholesterol** (Level 3 patients have ~76% disease rate compared to ~44% for Level 1)
   - **Weight / BMI** (higher body mass correlates with elevated risk)
4. **Data Cleaning Required**:
   - Outlier blood pressures (e.g. values exceeding 250 mmHg or sub-zero) must be filtered.
   - Cases where systolic BP < diastolic BP must be removed.
   - Duplicate rows should be removed prior to model training.